# Day 53 — Baseline LR + cost framing

Status: COMPLETE. Balanced Logistic Regression in a leak-free pipeline:
ROC-AUC 0.682, PR-AUC 0.333. Costs defined up front ($150 FN / $5 FP) —
at 0.5 the bill is $301k, 90% of it missed no-shows. 10/10 tests pass.

In [1]:
import json
from pathlib import Path

m = json.loads(Path("../models/baseline_metrics.json").read_text())
print(f"train {m['n_train']} / test {m['n_test']} rows | "
      f"ROC-AUC {m['roc_auc']} | PR-AUC {m['pr_auc']}")
print(f"F1@0.5 {m['f1_at_0.5']} | confusion tn=11938 fp=5704 fn=1819 tp=2645")
c = m["cost_at_0.5"]
print(f"cost@0.5: ${c['total']:,.0f} = {c['fn']} x $150 (FN) + {c['fp']} x $5 (FP)")
print("read: FNs cost 30x FPs, so 0.5 cannot be optimal — Day 54 tunes on $.")

train 88,420 / test 22,106 rows | ROC-AUC 0.6817 | PR-AUC 0.3333
F1@0.5 0.4129 | confusion tn=11938 fp=5704 fn=1819 tp=2645
cost@0.5: $301,370 = 1,819 x $150 (FN) + 5,704 x $5 (FP)
read: FNs cost 30x FPs, so 0.5 cannot be optimal — Day 54 tunes on $.


## Findings

1. **The floor ranks modestly (0.68) but the bill already frames everything**:
   $273k of $301k is missed no-shows. Any threshold work that doesn't move
   FN is theater.
2. **Costs before thresholds, always**: the 30:1 ratio was fixed with
   stakeholders-style assumptions *before* seeing a single probability —
   otherwise every threshold choice silently smuggles in 1:1 thinking.
3. **PR-AUC 0.33 vs 0.20 base rate**: real signal above chance; Day 54's trees
   must lift ranking while Day 54's threshold minimizes the bill.